In [ ]:
# Cell 1：导入依赖包
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei']
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# Cell 2：读取数据集
# 数据集放到 data/raw/UserBehavior.csv
columns = ['user_id', 'item_id', 'category_id', 'behavior_type', 'timestamp']
df = pd.read_csv('../data/raw/UserBehavior.csv', names=columns)

In [ ]:
# Cell3：基础信息查看
print("数据集形状：", df.shape)
print("\n前5行：")
print(df.head())
print("\n数据类型：")
print(df.dtypes)
print("\n缺失值统计：")
print(df.isnull().sum())
print("\n重复行数量：")
print(df.duplicated().sum())

In [ ]:
# Cell4：数据清洗 - 时间戳转日期时间
df['datetime'] = pd.to_datetime(df['timestamp'], unit='s')
df['date'] = df['datetime'].dt.date
df['hour'] = df['datetime'].dt.hour

# 行为类型映射：pv浏览、cart加购、fav收藏、buy购买
behavior_map = {'pv':'浏览', 'cart':'加购', 'fav':'收藏', 'buy':'购买'}
df['behavior_cn'] = df['behavior_type'].map(behavior_map)

# 剔除异常时间（官方数据集范围：2017-11-25 ~ 2017-12-03）
start_date = pd.to_datetime('2017-11-25')
end_date = pd.to_datetime('2017-12-04')
df = df[(df['datetime'] >= start_date) & (df['datetime'] < end_date)]
print("清洗后数据量：", df.shape)

In [ ]:
# Cell5：行为分布统计 & 漏斗计算
behavior_cnt = df['behavior_type'].value_counts()
print("各行为数量：")
print(behavior_cnt)

pv = behavior_cnt['pv']
cart = behavior_cnt['cart']
fav = behavior_cnt['fav']
buy = behavior_cnt['buy']

print(f"\n浏览→加购转化率：{cart/pv:.2%}")
print(f"浏览→收藏转化率：{fav/pv:.2%}")
print(f"浏览→购买转化率：{buy/pv:.2%}")
print(f"加购+收藏→购买转化率：{buy/(cart+fav):.2%}")

In [ ]:
# Cell6：用户活跃小时分布
hour_count = df.groupby('hour')['user_id'].nunique()
plt.figure(figsize=(10,4))
sns.barplot(x=hour_count.index, y=hour_count.values)
plt.title('用户活跃时段分布（独立用户数）')
plt.xlabel('小时')
plt.ylabel('独立用户数')
plt.savefig('../img/hour_active.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Cell7：转化漏斗图数据（后面可以放到streamlit看板）
funnel_data = pd.DataFrame({
    'stage': ['浏览','加购','收藏','购买'],
    'count': [pv, cart, fav, buy]
})
funnel_data